<a href="https://colab.research.google.com/github/vishnupriya1002/An-Automated-student-Analysis-System-with-CGPA-Computation-using-Predictive-Analysis-Algorithm/blob/main/Copy_of_Vishnupriya_T_612923101122_Advanced_RAG_Government_Scheme_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Vishnupriya T (612923101122)
## RAG-Based Intelligent Government Scheme Discovery and Eligibility Verification System

**Google Colab project notebook**

This notebook builds a Retrieval-Augmented Generation (RAG) prototype that:
- indexes government-scheme information from a starter knowledge base and user-uploaded PDF documents;
- retrieves relevant passages using sentence embeddings and FAISS;
- checks profile details against structured eligibility rules where those rules are available;
- produces a grounded response with source references; and
- provides an optional Gradio interface.

> **Important:** This is an academic prototype, not an official government eligibility decision. Scheme rules, income limits, dates, application windows and links can change. Verify every result with the relevant official government portal before applying. The starter records are illustrative seed data; replace or validate them against current official scheme documents before real-world use. Do not enter highly sensitive personal information.

## 1. Runtime setup

In Colab, select **Runtime → Change runtime type**. A GPU is helpful for optional language-model generation but is not mandatory for retrieval and eligibility-rule matching.

In [ ]:
!pip -q install -U sentence-transformers faiss-cpu pymupdf gradio pandas numpy

In [ ]:
import os, re, json, textwrap, warnings
from pathlib import Path
from typing import List, Dict, Any
import numpy as np
import pandas as pd
import fitz
import faiss
import gradio as gr
from sentence_transformers import SentenceTransformer

warnings.filterwarnings("ignore")
print("Environment ready.")

Environment ready.


## 2. Configure the Gemini API securely

This project uses Google's official `google-genai` Python SDK. In Google Colab, add your key in the **Secrets** panel (key icon on the left) using the secret name `GEMINI_API_KEY`, then enable notebook access for that secret.

Get or create an API key in [Google AI Studio](https://aistudio.google.com/app/apikey). **Never paste your key into a code cell or share it in a public notebook.** The notebook reads it from Colab Secrets at runtime.

In [ ]:
# GEMINI API CONFIGURATION

!pip install -q -U google-genai

from google import genai
from getpass import getpass

GEMINI_MODEL = "gemini-2.5-flash"

# Enter API key securely
GEMINI_API_KEY = getpass("Enter your Gemini API Key: ")

# Initialize Gemini
client = genai.Client(api_key=GEMINI_API_KEY)

print("Gemini API connected successfully!")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.1 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.1 which is incompatible.
Enter your Gemini API Key: ··········
Gemini API connected successfully!


## 2. Create the starter scheme knowledge base

The starter records below demonstrate the data structure and retrieval workflow. They are not a complete or authoritative catalogue. Add official scheme PDFs in the upload step and validate all structured eligibility fields against the latest official guidelines.

In [ ]:
SCHEMES = [
    {
        "scheme_id": "PM-KISAN-DEMO",
        "name": "Pradhan Mantri Kisan Samman Nidhi (PM-KISAN)",
        "level": "Central",
        "benefit": "Income support to eligible landholding farmer families, subject to scheme rules and exclusions.",
        "eligibility_text": "Eligible landholding farmer families, subject to applicable exclusions and verification in official records.",
        "occupation": ["farmer", "agriculture"],
        "age_min": None, "age_max": None,
        "income_max": None,
        "state": "All India",
        "documents": ["Aadhaar details", "landholding records", "bank account details", "other documents required by the official process"],
        "application": "Check the official PM-KISAN portal and local agriculture office for current registration and status procedures.",
        "official_url": "https://pmkisan.gov.in/",
        "source_note": "Starter record; verify current eligibility and exclusions on the official portal."
    },
    {
        "scheme_id": "NSP-DEMO",
        "name": "National Scholarship Portal schemes (NSP)",
        "level": "Central",
        "benefit": "Scholarships for eligible students under individual central and centrally sponsored schemes.",
        "eligibility_text": "Eligibility varies by scholarship, course, institution, category and family income. Each scheme has its own current guidelines.",
        "occupation": ["student"],
        "age_min": None, "age_max": None,
        "income_max": None,
        "state": "All India",
        "documents": ["Student identity and enrolment details", "income certificate if required", "bank details", "category certificate if applicable"],
        "application": "Review the current scholarship guidelines and apply through the National Scholarship Portal when the relevant application window is open.",
        "official_url": "https://scholarships.gov.in/",
        "source_note": "Eligibility must be checked against the specific scholarship's current guidelines."
    },
    {
        "scheme_id": "TN-UNEMP-DEMO",
        "name": "Tamil Nadu Employment and Career Guidance Services",
        "level": "Tamil Nadu",
        "benefit": "Employment registration, career guidance and access to relevant employment services; specific benefits depend on the service or programme.",
        "eligibility_text": "Requirements vary by service and programme. Check current Tamil Nadu government instructions.",
        "occupation": ["job seeker", "unemployed", "student"],
        "age_min": None, "age_max": None,
        "income_max": None,
        "state": "Tamil Nadu",
        "documents": ["Identity and education details", "other documents specified by the service"],
        "application": "Consult the official Tamil Nadu employment portal for current services and procedures.",
        "official_url": "https://employment.tn.gov.in/",
        "source_note": "Starter record; this is not a single guaranteed cash-benefit scheme."
    }
]

scheme_df = pd.DataFrame(SCHEMES)
display(scheme_df[["scheme_id","name","level","official_url"]])

,scheme_id,name,level,official_url
0,PM-KISAN-DEMO,Pradhan Mantri Kisan Samman Nidhi (PM-KISAN),Central,https://pmkisan.gov.in/
1,NSP-DEMO,National Scholarship Portal schemes (NSP),Central,https://scholarships.gov.in/
2,TN-UNEMP-DEMO,Tamil Nadu Employment and Career Guidance Serv...,Tamil Nadu,https://employment.tn.gov.in/


## 3. Add official scheme documents (PDF)

Upload government scheme guidelines, notifications or official FAQs as PDFs. The notebook extracts text and keeps the filename as a source label. Prefer documents downloaded from official government domains. Scanned PDFs may need OCR; this notebook reports when little text is extracted.

In [ ]:
from google.colab import files

uploaded = files.upload()  # Select one or more official PDF files
pdf_documents = []

for filename in uploaded.keys():
    if filename.lower().endswith(".pdf"):
        doc = fitz.open(filename)
        pages = []
        for page_no, page in enumerate(doc, start=1):
            page_text = page.get_text("text")
            if page_text.strip():
                pages.append(f"[Page {page_no}] {page_text}")
        full_text = "\\n".join(pages)
        if len(full_text.strip()) < 100:
            print(f"Warning: very little selectable text found in {filename}. It may be a scanned PDF; OCR may be required.")
        pdf_documents.append({"source": filename, "text": full_text})
    else:
        print(f"Skipped non-PDF file: {filename}")

print(f"Loaded {len(pdf_documents)} PDF document(s).")

## 4. Prepare chunks for retrieval

Long documents are split into overlapping text chunks so that relevant sections can be retrieved without sending entire PDFs to the language model.

In [ ]:
def chunk_text(text: str, chunk_size: int = 900, overlap: int = 150) -> List[str]:
    text = re.sub(r"\\s+", " ", text).strip()
    if not text:
        return []
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        if end == len(text):
            break
        start = max(end - overlap, start + 1)
    return chunks

documents = []
for item in SCHEMES:
    body = (
        f"Scheme: {item['name']}. Level: {item['level']}. "
        f"Benefit: {item['benefit']} Eligibility: {item['eligibility_text']} "
        f"Documents: {', '.join(item['documents'])}. "
        f"Application: {item['application']} Official URL: {item['official_url']}. "
        f"Note: {item['source_note']}"
    )
    for i, chunk in enumerate(chunk_text(body)):
        documents.append({"text": chunk, "source": item["name"], "chunk": i})

for pdf in pdf_documents:
    for i, chunk in enumerate(chunk_text(pdf["text"])):
        documents.append({"text": chunk, "source": pdf["source"], "chunk": i})

print("Total indexed chunks:", len(documents))

## 5. Generate embeddings and build the FAISS index

The `all-MiniLM-L6-v2` sentence-transformer creates compact semantic embeddings. FAISS performs efficient nearest-neighbour search over those vectors.

In [ ]:
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)

texts = [d["text"] for d in documents]
embeddings = embedder.encode(
    texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")

dimension = embeddings.shape[1]
vector_index = faiss.IndexFlatIP(dimension)  # cosine similarity because vectors are normalized
vector_index.add(embeddings)

print(f"FAISS index built with {vector_index.ntotal} vectors.")

## 6. Semantic retrieval

The retriever returns the most relevant chunks and their source labels. The top-k value can be adjusted for experimentation.

In [ ]:
def retrieve(query: str, top_k: int = 5) -> List[Dict[str, Any]]:
    query = (query or "").strip()
    if not query:
        return []
    q_vec = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype("float32")
    k = min(max(1, int(top_k)), len(documents))
    scores, indices = vector_index.search(q_vec, k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx >= 0:
            item = documents[int(idx)].copy()
            item["score"] = float(score)
            results.append(item)
    return results

for result in retrieve("scholarship for students and required documents", top_k=3):
    print(round(result["score"], 3), result["source"], ":", result["text"][:220], "...")

## 7. Structured eligibility matching

Eligibility is only marked as **potentially matching**, **not enough information**, or **not matched by the available structured rules**. Missing rules are never treated as proof of eligibility. For schemes with complex criteria, consult the official document and relevant authority.

In [ ]:
def normalize(value):
    return str(value or "").strip().lower()

def check_scheme(profile: Dict[str, Any], scheme: Dict[str, Any]) -> Dict[str, Any]:
    reasons, missing, failed = [], [], []

    age = profile.get("age")
    if scheme.get("age_min") is not None:
        if age is None: missing.append("age")
        elif int(age) < scheme["age_min"]: failed.append(f"age must be at least {scheme['age_min']}")
    if scheme.get("age_max") is not None:
        if age is None: missing.append("age")
        elif int(age) > scheme["age_max"]: failed.append(f"age must be at most {scheme['age_max']}")

    income = profile.get("annual_income")
    if scheme.get("income_max") is not None:
        if income is None: missing.append("annual family income")
        elif float(income) > scheme["income_max"]:
            failed.append(f"annual income exceeds the configured limit of ₹{scheme['income_max']:,}")

    occupation = normalize(profile.get("occupation"))
    allowed = [normalize(x) for x in scheme.get("occupation", [])]
    if allowed:
        if not occupation: missing.append("occupation")
        elif not any(x in occupation or occupation in x for x in allowed):
            failed.append("occupation does not match the configured category")

    user_state = normalize(profile.get("state"))
    scheme_state = normalize(scheme.get("state"))
    if scheme_state not in ("all india", ""):
        if not user_state: missing.append("state")
        elif user_state not in scheme_state and scheme_state not in user_state:
            failed.append(f"scheme is listed for {scheme['state']}")

    if failed:
        status = "Not matched by the available rules"
        reasons.extend(failed)
    elif missing:
        status = "Not enough information"
        reasons.append("Additional details or official scheme-specific checks are required.")
    else:
        status = "Potential match — verify with official authority"
        reasons.append("No conflict was found in the limited structured rules available in this prototype.")
    return {"status": status, "reasons": reasons, "missing": sorted(set(missing))}

def find_matching_schemes(profile):
    output = []
    for scheme in SCHEMES:
        result = check_scheme(profile, scheme)
        output.append({"scheme": scheme, **result})
    return output

## 8. RAG answer generation with retrieved evidence

The default answer is evidence-grounded and template-based, which keeps the notebook usable without a paid API key. An optional small open-source text-generation model can be loaded in the next cell. Always inspect the cited source passages because language models can make mistakes.

In [ ]:
def build_grounded_answer(question: str, profile: Dict[str, Any] = None, top_k: int = 5) -> str:
    profile = profile or {}
    hits = retrieve(question, top_k=top_k)
    if not hits:
        return "I could not find relevant information in the current knowledge base. Add official scheme documents or rephrase your question."

    context = "\\n\\n".join(
        f"Source: {h['source']} (similarity {h['score']:.3f})\\n{h['text']}"
        for h in hits
    )
    eligibility = find_matching_schemes(profile) if profile else []

    answer = [
        "RETRIEVED INFORMATION",
        "The following passages were retrieved from the indexed knowledge base:",
        context,
    ]
    if eligibility:
        answer.extend(["\\nPRELIMINARY ELIGIBILITY SCREENING"])
        for item in eligibility:
            answer.append(f"- {item['scheme']['name']}: {item['status']}")
            for reason in item["reasons"]:
                answer.append(f"  • {reason}")
            if item["missing"]:
                answer.append("  • Missing details: " + ", ".join(item["missing"]))
            answer.append(f"  • Official source: {item['scheme']['official_url']}")
    answer.append(
        "\\nDISCLAIMER: This is an informational screening result, not an official eligibility decision. "
        "Scheme rules may change. Confirm the latest guidelines and final eligibility with the official portal or department."
    )
    return "\\n".join(answer)

sample_profile = {
    "age": 21,
    "annual_income": 180000,
    "occupation": "student",
    "state": "Tamil Nadu"
}
print(build_grounded_answer("What scholarship options and documents are relevant for students?", sample_profile))

## 9. Gemini-powered RAG response generation

Retrieved passages are passed to Gemini as grounding context. The system instruction tells Gemini not to invent scheme rules and to state when the supplied sources do not contain an answer. Retrieved source labels are included for verification.

In [ ]:
SYSTEM_INSTRUCTION = (
    "You are an informational assistant for discovering Indian central and Tamil Nadu government schemes. "
    "Use only the supplied retrieved context to answer factual questions about schemes. "
    "Do not invent scheme names, eligibility rules, income limits, deadlines, benefits, document requirements, "
    "application status, or official URLs. If retrieved context is insufficient, say the information is not "
    "available in the indexed sources and recommend checking the official government department. "
    "Distinguish preliminary screening from an official eligibility decision."
)

def generate_with_gemini(question: str, top_k: int = 5, profile: Dict[str, Any] = None) -> str:
    hits = retrieve(question, top_k=top_k)
    if not hits:
        return "No relevant information was retrieved. Upload current official scheme documents or try another question."

    context = "\\n\\n".join(
        f"[Source: {h['source']} | Similarity: {h['score']:.3f}]\\n{h['text']}"
        for h in hits
    )
    safe_profile = {
        "age": (profile or {}).get("age"),
        "annual_income": (profile or {}).get("annual_income"),
        "occupation": (profile or {}).get("occupation"),
        "state": (profile or {}).get("state")
    }
    prompt = (
        f"User question: {question}\\n"
        f"User-provided screening details (may be incomplete): {json.dumps(safe_profile, ensure_ascii=False)}\\n\\n"
        f"Retrieved knowledge-base context:\\n{context}\\n\\n"
        "Answer using only this context. If a criterion is missing or unclear, state that it cannot be confirmed. "
        "Do not infer eligibility from missing data. Mention relevant source names and official URLs when present. "
        "End with a reminder to verify current details on official portals."
    )

    response = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
        config=genai.types.GenerateContentConfig(
            system_instruction=SYSTEM_INSTRUCTION,
            temperature=0.2,
            max_output_tokens=900
        )
    )
    answer = (response.text or "").strip()
    source_list = "\\n".join(f"- {h['source']}" for h in hits)
    return (
        (answer if answer else "Gemini returned an empty response.") +
        "\\n\\nRetrieved source passages:\\n" + source_list +
        "\\n\\nDisclaimer: This is informational assistance, not an official eligibility decision."
    )

# API smoke test: requires a valid GEMINI_API_KEY in Colab Secrets.
print(generate_with_gemini(
    "What is the National Scholarship Portal and where can students verify scheme details?",
    top_k=3
))

## 10. End-to-end user function

Enter only the information needed for an initial screening. Leave unknown fields blank. This function combines semantic retrieval with the structured rules and provides official links for the starter records.

In [ ]:
def scheme_assistant(question, age=None, annual_income=None, occupation="", state=""):
    profile = {
        "age": int(age) if age not in (None, "") else None,
        "annual_income": float(annual_income) if annual_income not in (None, "") else None,
        "occupation": occupation,
        "state": state
    }

    # Gemini generates the natural-language response from retrieved RAG context.
    rag_text = generate_with_gemini(question, top_k=5, profile=profile)

    # Deterministic rule checks are displayed separately.
    eligibility = find_matching_schemes(profile)
    eligible_text = []
    for item in eligibility:
        eligible_text.append(
            f"**{item['scheme']['name']} — {item['status']}**\\n"
            f"Benefits: {item['scheme']['benefit']}\\n"
            f"Documents: {', '.join(item['scheme']['documents'])}\\n"
            f"Application: {item['scheme']['application']}\\n"
            f"Official URL: {item['scheme']['official_url']}\\n"
            f"Reason(s): {'; '.join(item['reasons'])}\\n"
        )

    return (
        f"### Gemini-powered RAG response\\n{rag_text}\\n\\n"
        f"### Preliminary structured screening\\n" + "\\n".join(eligible_text) +
        "\\n**Reminder:** Confirm all details with the official department. This prototype does not make an official decision."
    )

print(scheme_assistant(
    "Find student scholarship information and application documents",
    age=21, annual_income=180000, occupation="student", state="Tamil Nadu"
))

## 11. Interactive Gradio application

Launches a shareable temporary Gradio link for demonstration. The link is temporary and should not be used to collect real sensitive personal information. For a persistent public deployment, host the app securely and review privacy, access control and data-retention requirements.

In [ ]:
demo = gr.Interface(
    fn=scheme_assistant,
    inputs=[
        gr.Textbox(label="Your question", placeholder="Example: Find schemes for students in Tamil Nadu"),
        gr.Number(label="Age (optional)", precision=0),
        gr.Number(label="Annual family income in INR (optional)", precision=0),
        gr.Textbox(label="Occupation (optional)", placeholder="student / farmer / job seeker"),
        gr.Textbox(label="State (optional)", placeholder="Tamil Nadu")
    ],
    outputs=gr.Markdown(label="Scheme discovery and preliminary screening"),
    title="Gemini RAG Government Scheme Discovery and Eligibility Assistant",
    description="Gemini-powered RAG retrieval and preliminary eligibility screening. Verify all results with official government sources."
)

demo.launch(share=True, debug=False)

## 12. Basic retrieval evaluation

This simple smoke test checks whether expected scheme topics appear among retrieved results. It is a small demonstration, not a rigorous benchmark. For a research evaluation, prepare a larger human-reviewed question set and measure Recall@k, MRR, answer faithfulness and eligibility-screening accuracy.

In [ ]:
test_cases = [
    ("student scholarship portal", "National Scholarship Portal schemes (NSP)"),
    ("income support for farmers", "Pradhan Mantri Kisan Samman Nidhi (PM-KISAN)"),
    ("Tamil Nadu employment services", "Tamil Nadu Employment and Career Guidance Services")
]

evaluation_rows = []
for query, expected_source in test_cases:
    results = retrieve(query, top_k=5)
    retrieved_sources = [r["source"] for r in results]
    evaluation_rows.append({
        "query": query,
        "expected_source": expected_source,
        "retrieved_sources": retrieved_sources,
        "hit_at_5": expected_source in retrieved_sources
    })

evaluation_df = pd.DataFrame(evaluation_rows)
display(evaluation_df)
print("Hit@5:", round(evaluation_df["hit_at_5"].mean(), 3))

## 13. Save the processed knowledge base and index

This cell saves the FAISS index and source metadata to the Colab session. Download the files before closing the runtime, or save them to your own Google Drive.

In [ ]:
faiss.write_index(vector_index, "government_schemes.faiss")
with open("government_schemes_metadata.json", "w", encoding="utf-8") as f:
    json.dump(documents, f, ensure_ascii=False, indent=2)

print("Saved government_schemes.faiss and government_schemes_metadata.json")

In [ ]:
from google.colab import files
files.download("government_schemes.faiss")
files.download("government_schemes_metadata.json")

## 14. Suggested future enhancements

1. Replace the illustrative starter records with a verified, regularly updated collection of official central and state scheme guidelines.
2. Add OCR for scanned PDFs and multilingual document support.
3. Implement hybrid retrieval (BM25 + dense vectors) and a cross-encoder reranker.
4. Store scheme criteria in a validated structured database with effective dates and version history.
5. Add query rewriting, query decomposition and self-checking retrieval.
6. Add multilingual user interaction for English and Tamil.
7. Evaluate retrieval and generation with a manually verified benchmark.
8. Add privacy safeguards, consent, minimal data collection and secure hosting.

### Final note

A positive result means only that the prototype did not find a conflict in its limited configured rules. It does **not** establish legal entitlement. Missing information, unmodelled exclusions and outdated documents can change the outcome. Always rely on the latest official scheme notification and the responsible government department.